In [1]:
import pandas as pd

In [3]:

# Base customer file
customers_df = pd.read_csv(
    "../data/processed/customer_intelligence.csv"
)

# Segment file
segments_df = pd.read_csv(
    "../data/processed/rfm_with_clusters.csv"
)

# CLV file
clv_df = pd.read_csv(
    "../data/processed/rfm_with_CLVclusters.csv"
)

In [4]:
segments_df = segments_df[
    [
        "CustomerID",
        "Cluster_Label"
    ]
].rename(
    columns={
        "CustomerID": "customer_id",
        "Cluster_Label": "segment"
    }
)

In [5]:
print(clv_df.columns.tolist())

['Customer ID', 'Frequency', 'Monetary', 'First Purchase Date', 'Last Purchase Date', 'Lifespan', 'Cluster']


In [6]:
clv_df = clv_df[
    [
        "Customer ID",
        "Cluster"
    ]
].rename(
    columns={
        "Customer ID": "customer_id",
        "Cluster": "clv_tier"
    }
)

In [7]:
customers_df["customer_id"] = customers_df["customer_id"].astype(float)

segments_df["customer_id"] = segments_df["customer_id"].astype(float)

clv_df["customer_id"] = clv_df["customer_id"].astype(float)

In [8]:
customers_df = customers_df.merge(
    segments_df,
    on="customer_id",
    how="left"
)

In [9]:
customers_df = customers_df.merge(
    clv_df,
    on="customer_id",
    how="left"
)

In [10]:
print(customers_df.head())

print(
    customers_df[
        ["customer_id", "segment", "clv_tier"]
    ].head()
)

   customer_id  frequency  monetary  lifespan  segment_x  clv_tier_x  \
0      12346.0         12  77556.46       400        NaN           1   
1      12347.0          8   4921.53       402        NaN           1   
2      12348.0          5   2019.40       362        NaN           1   
3      12349.0          4   4428.69       570        NaN           1   
4      12350.0          1    334.40         0        NaN           0   

   churn_probability  first_purchase_date   last_purchase_date  \
0                NaN  2009-12-14 08:34:00  2011-01-18 10:01:00   
1                NaN  2010-10-31 14:20:00  2011-12-07 15:52:00   
2                NaN  2010-09-27 14:59:00  2011-09-25 13:13:00   
3                NaN  2010-04-29 13:20:00  2011-11-21 09:51:00   
4                NaN  2011-02-02 16:01:00  2011-02-02 16:01:00   

                 last_updated          segment_y  clv_tier_y  
0  2026-06-05 19:43:02.858787    Loyal Customers           1  
1  2026-06-05 19:43:02.858787  Dormant Custo

KeyError: "['segment', 'clv_tier'] not in index"

In [11]:
customers_df = customers_df.drop(
    columns=[
        "segment_x",
        "clv_tier_x"
    ]
)

customers_df = customers_df.rename(
    columns={
        "segment_y": "segment",
        "clv_tier_y": "clv_tier"
    }
)

In [12]:
customers_df[
    ["customer_id", "segment", "clv_tier"]
].head()

,customer_id,segment,clv_tier
0,12346.0,Loyal Customers,1
1,12347.0,Dormant Customers,1
2,12348.0,Dormant Customers,1
3,12349.0,Dormant Customers,1
4,12350.0,Dormant Customers,0


In [13]:
print(customers_df.columns.tolist())

['customer_id', 'frequency', 'monetary', 'lifespan', 'churn_probability', 'first_purchase_date', 'last_purchase_date', 'last_updated', 'segment', 'clv_tier']


In [15]:
churn_df = pd.read_csv(
    "../data/processed/churn_probabilities.csv"
)

In [ ]:
churn_df = churn_df.rename(
    columns={
        "CustomerID": "customer_id"
    }
)

In [17]:
customers_df = customers_df.merge(
    churn_df,
    on="customer_id",
    how="left"
)

In [19]:
customers_df = customers_df.drop(
    columns=["churn_probability_x"]
)

customers_df = customers_df.rename(
    columns={
        "churn_probability_y":
        "churn_probability"
    }
)

In [20]:
print(
    customers_df[
        [
            "customer_id",
            "churn_probability"
        ]
    ].head()
)

   customer_id  churn_probability
0      12346.0           0.004696
1      12347.0           0.037968
2      12348.0           0.117543
3      12349.0           0.123847
4      12350.0           0.450300


In [21]:
print(customers_df.columns.tolist())

['customer_id', 'frequency', 'monetary', 'lifespan', 'first_purchase_date', 'last_purchase_date', 'last_updated', 'segment', 'clv_tier', 'churn_probability']


In [23]:
customers_df['clv_tier'].unique()

array([1, 0, 2], dtype=int64)

In [24]:
customers_df["clv_tier"] = customers_df["clv_tier"].map({
    0: "Low CLV",
    1: "Medium CLV",
    2: "High CLV"
})

In [25]:
customers_df.to_csv(
    "../data/processed/customer_intelligence.csv",
    index=False
)